# Sampling important IMAGE tokens on SmolVLM2

Chains the full pipeline on the real **`HuggingFaceTB/SmolVLM2-2.2B-Instruct`**:

```
raw text->vision attention  ->  rater_selection (important TEXT tokens)
                            ->  visual_selection (important IMAGE tokens, over ALL layers)
```

Then it (a) runs the `visual_selection` invariant checks, (b) prints the
diagram's three test-case numbers, and (c) shows the selected image patches as a
heatmap over the image.

> **Runtime:** GPU runtime. SmolVLM2 is open (no token needed).

## 1. Install + clone

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words pytest matplotlib

In [ ]:
!rm -rf text_vision_attention_map          # fresh checkout
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git
%cd text_vision_attention_map

## 2. Probe SmolVLM, then rater_selection -> visual_selection
One forward captures raw attention for ALL decoder layers; we pick the important
text tokens, then use them to sample the important image tokens over all layers.

In [ ]:
import importlib.util, os

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
V = _load("test_visual_selection", "tests/test_visual_selection.py")
import rater_selection as RS
import visual_selection as VS

o = S.make_smolvlm_output()          # raw text->vision attention, ALL decoder layers
assert o is not None, "SmolVLM probe failed to load — see printed reason above."

from transformers import AutoProcessor
tokenizer = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-2.2B-Instruct").tokenizer

maps_per_head, tpos, vpos = RS.sliced_maps_from_full(
    o.raw_scores, o.image_token_mask, o.text_token_mask)
text_tokens = tokenizer.convert_ids_to_tokens(o.input_ids[tpos].tolist())

# --- step 1: important TEXT tokens (raters) ---
rres = RS.select_important_text_tokens(
    maps_per_head, text_tokens=text_tokens, tokenizer=tokenizer,
    question=o.question, pct=0.5)

# --- step 2: important IMAGE tokens (over ALL layers) ---
vres = VS.select_from_rater(maps_per_head, rres, pct=0.5)

print("question           :", repr(o.question))
print("rater text tokens  :", rres.kept_tokens(text_tokens))
print("decoder layers used:", len(vres.band), "->", vres.band[:4], "...", vres.band[-2:])
print("image tokens L_v    :", vres.L_v)
print("kept image tokens   :", vres.n_kept, f"(top {vres.n_kept}/{vres.L_v}, pct={vres.pct})")

## 3. Run the visual_selection invariant checks

In [ ]:
case = V.make_case(maps_per_head, rres.rater_mask, pct=0.5, name="smolvlm")
passed = 0
for fn in V.ALL_CHECKS:
    try:
        fn(case)
        print(f"PASS  {fn.__name__}")
        passed += 1
    except AssertionError as e:
        print(f"FAIL  {fn.__name__}: {e}")
print(f"\n{passed}/{len(V.ALL_CHECKS)} visual-selection checks passed on SmolVLM2.")

## 4. The diagram's three test cases — concrete numbers

In [ ]:
# (1) final distribution: row axis = 1, column axis = #image tokens
print("final importance shape :", tuple(vres.importance.shape),
      "-> as row:", tuple(vres.importance.view(1, -1).shape))
print("cols == #image tokens  :", vres.importance.shape[0] == vres.L_v)

# (2) #prob distributions before the layer-sum == #decoder layers
print("per-layer distributions:", vres.per_layer.shape[0], "== #layers:", len(vres.band))

# (3) input matrix rows == #rater text tokens (after threshold)
print("input rows (raters)    :", vres.n_raters, "== kept text tokens:", int(rres.rater_mask.sum()))

## 5. Heatmap: where the selected image tokens are
The `L_v` image tokens form a grid; we reshape the importance over that grid and
overlay it on the image, then show the kept (top-k) patches.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

img = S._load_demo_image()                 # same image the probe used
L_v = vres.L_v
g = int(round(math.sqrt(L_v)))
imp = vres.importance.numpy()
msk = vres.vision_mask.numpy().astype(np.float32)

if g * g != L_v:                            # pad to a square grid if needed
    g = math.ceil(math.sqrt(L_v)); pad = g * g - L_v
    imp = np.concatenate([imp, np.zeros(pad, imp.dtype)])
    msk = np.concatenate([msk, np.zeros(pad, msk.dtype)])
heat = imp.reshape(g, g)
kept = msk.reshape(g, g)
print(f"image-token grid: {g} x {g}  (L_v={L_v})")

def _up(a, mode=Image.BILINEAR):
    a = (a / (a.max() + 1e-9) * 255).astype('uint8')
    return np.array(Image.fromarray(a).resize(img.size, mode))

fig, ax = plt.subplots(1, 3, figsize=(16, 6))
ax[0].imshow(img); ax[0].set_title("image"); ax[0].axis("off")
ax[1].imshow(img); ax[1].imshow(_up(heat), cmap="jet", alpha=0.5)
ax[1].set_title("image-token importance (all layers)"); ax[1].axis("off")
ax[2].imshow(img); ax[2].imshow(_up(kept, Image.NEAREST), cmap="Greens", alpha=0.45)
ax[2].set_title(f"kept image tokens (top {vres.n_kept}/{L_v})"); ax[2].axis("off")
plt.tight_layout(); plt.show()

## 6. (Optional) sweep pct — how many image tokens survive

In [ ]:
for pct in (0.0, 0.25, 0.5, 0.75, 0.9):
    r = VS.select_important_image_tokens(maps_per_head, rres.rater_mask, pct=pct)
    print(f"pct={pct:<4}  kept image tokens = {r.n_kept:>3} / {r.L_v}")